# Subsetting, the retrospective archive, and what's deferred

A plain operational request downloads whole-CONUS files. A **subset**
(`sites=` / bbox) or the **retrospective** archive is *read* — not
downloaded whole — through pyramids' `LabeledDataset` reader
(pyramids ≥ 0.29.0). earthlens never imports `xarray`/`zarr` itself.

For the **tabular** products (`chrtout`, `lakeout`, `coastal`) the
reader opens the store anonymously + lazily, slices, and writes a tidy
`feature_id × time` **Parquet** table. This notebook does one **live**
retrospective subset and shows what is still deferred.

## Live: retrospective streamflow for three reaches

Opens the 1.4 TB retrospective `chrtout.zarr` anonymously, slices to
three `feature_id`s and a two-day window, and writes a Parquet table —
no whole-store download.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

from earthlens import EarthLens

nwm = EarthLens(
    data_source="nwm",
    start='2010-06-01',
    end='2010-06-02',
    dataset='chrtout',
    variables=['streamflow'],
    aoi=[-180, -90, 180, 90],
    configuration='analysis_assim',
    mode='retrospective',
    sites=[101, 179, 181],
    path=tempfile.mkdtemp(prefix='nwm_retro_'),
)
paths = nwm.download(progress_bar=False)
df = pd.read_parquet(paths[0])
print(Path(paths[0]).name, '|', df.shape)
df.head()

The result is a tidy table — one row per `(feature_id, time)` — with
the streamflow plus the in-file `latitude`/`longitude`/`gage_id` coords.

In [ ]:
print('feature_ids:', sorted(df['feature_id'].unique().tolist()))
print('columns:', list(df.columns))

## USGS `gage_id` join

`sites=` also accepts USGS `gage_id` **strings** — earthlens routes
those to the in-file `gage_id` coordinate join instead of `feature_id`.

In [ ]:
# (enumerated, not downloaded here) — gage_id strings vs feature_id ints
g = EarthLens(
    data_source="nwm",
    start='2010-06-01',
    end='2010-06-02',
    dataset='chrtout',
    variables=['streamflow'],
    aoi=[-180, -90, 180, 90],
    configuration='analysis_assim',
    mode='retrospective',
    sites=[101, '01010000'],
)
print('feature_id ints:', g._feature_ids(), '| gage_id strings:', g._gage_ids())

## Still deferred: subsetting the **gridded** products

`ldasout` / `rtout` / `forcing` are gridded; subsetting them (or reading
their retrospective cube) needs a gridded cloud-cube reader pyramids
does not yet expose, so it raises a clear `NotImplementedError`. Their
**whole-file** operational download works.

In [ ]:
def show(label, fn):
    try:
        fn()
        print(f'{label}: (no error)')
    except NotImplementedError as exc:
        print(f'{label}: NotImplementedError -> {str(exc)[:70]}...')


grid = EarthLens(
    data_source="nwm",
    start='2010-06-01',
    end='2010-06-02',
    dataset='ldasout',
    variables=['SOIL_M'],
    aoi=[-180, -90, 180, 90],
    configuration='analysis_assim',
    mode='retrospective',
    sites=[101],
)
show('gridded retrospective subset', lambda: grid.download(progress_bar=False))

## `aggregate=` is rejected

`chrtout` is feature-id indexed (not griddable) and a gridded reduce
needs a separate reader, so the temporal aggregator is unsupported.

In [ ]:
agg = EarthLens(
    data_source="nwm",
    start='2010-06-01',
    end='2010-06-02',
    dataset='chrtout',
    variables=['streamflow'],
    aoi=[-180, -90, 180, 90],
    configuration='analysis_assim',
    mode='retrospective',
)
show('aggregate=', lambda: agg.download(aggregate=object()))